# 实验目的
通过模型退化，验证PE模型正确性，分段线性位移变换正确性。
1. 将海浪高度，风速等条件设置为0，验证模型能否完美还原双射线模型解
2. 在距离发射源100m处添加单个斜坡，验证PE与FDTD模型的均方根误差

# 实验步骤
## 验证方案
抛物方程（Parabolic Equation, PE）方法自20世纪40年代由Leontovich和Fock提出，并在70年代由Tappert引入分步傅里叶变换（SSFT）求解后，已经成为解决大尺度、非均匀介质（如大气波导）中电磁波传播的**“黄金标准” (Gold Standard)**。因此不需要验证PE算法。
但是引入分段线性位移变换修正下边界，这是对原有模型的改进，需要验证是否引入了非物理的数值误差，遮挡效应计算是否准确。但是不需要进行物理实测验证，而是使用数值对标验证。
### 退化验证
将海浪高度设为0（退化为平坦海面）。验证分段线性变换在平坦情况下是否能完美还原为标准PE的解（或双射线模型解）。如果平海面都算不对，说明变换矩阵或相位修正项推导有误。
### 与精确解进行对比
使用FDTD仿真全波解，FDTD直接求解麦克斯韦方程组，包含所有散射、衍射和反射效应，被视为全波解（Full-wave solution）。

1. 构建一个小尺度的海面模型（例如几百米，因为FDTD算不动几十公里）。

2. 设置一个确定的分段线性海浪形状。

3. 分别用改进PE和FDTD计算传播因子（Propagation Factor）。

4. 画出两条曲线：如果两者在远场吻合良好，且PE比FDTD快几个数量级，那么改进就是成功的。

### 指标
a. 均方根误差 (RMSE) -- 核心指标
$$RMSE = \sqrt{\frac{1}{N} \sum_{i=1}^{N} (L_{PE}(i) - L_{FDTD}(i))^2}$$
< 1 dB: 极好（Excellent），几乎完美复现全波解。

1 ~ 3 dB: 良好（Good），这是大多数改进型 PE 算法能达到的区间，完全可以接受。

\> 5 dB: 需要解释原因（例如只在深阴影区误差大，但在覆盖区很准）。


b. 平均偏差 (Mean Bias Error, MBE) —— 辅助指标
$$MBE = \frac{1}{N} \sum_{i=1}^{N} (L_{PE}(i) - L_{FDTD}(i))$$
意义：用于判断你的算法是否存在系统性误差。如果 MBE > 0，说明你的算法系统性地低估了损耗（过于乐观）。如果 MBE < 0，说明系统性地高估了损耗（过于保守）。理想情况下 MBE 应接近 0。

c. 最大绝对误差 (Max Absolute Error) —— 针对性指标
$$MaxError = \max |L_{PE}(i) - L_{FDTD}(i)|$$
意义：通常出现在干涉零点（Deep Nulls）。PE 方法在预测零点位置时通常会有轻微频移或位置偏移，导致该点误差巨大（例如 FDTD 是 -80dB，PE 是 -60dB）。如何辩解：如果最大误差只出现在极深的零点，你可以解释为“对于实际通信工程，低于接收机灵敏度（如 -110dBm）的深零点误差不影响连通性判断”。

# 方法延申
验证了方法正确性后，引入编队场景。用PE算出编队中N艘船的功率分布情况，生成一个图（Dynamic Graph）。聚焦于恶劣海况下编队构型的稳健性分析

# 代码实现
1. 生成JONSWAP海面
2. 得到观测海面高度（x-z平面）
3. 分别进行PE FDTD计算
4. 绘制对比图

In [22]:
# 导入包
import meep as mp
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import ImageGrid
import scipy.fft as fft
import pandas as pd
from abc import ABC, abstractmethod
import os
from scipy.ndimage import uniform_filter1d


# ==========================================
# 场景生成器 (Scene Generator)
# 定义全局物理参数、生成海面、统一下发坐标
# ==========================================
class SceneGenerator:
    def __init__(self, 
                 freq_ghz=0.3,      # 频率(GHz)
                 lx=320.0,          # 仿真域长度 (m) 
                 lz=50.0,           # 仿真域高度 (m)
                 dpml=5.0,          # 吸收层厚度 (m)
                 wind_speed=0.0,   # 风速 (m/s)
                 fetch_km=0.0,     # 风区 (km)
                 tx_height=5.0,     # 发射机高度 (相对海平面, m)
                 rx_height=5.0,
                 terrain_type='gaussian_hill'):    # 接收机观测高度 (相对海平面, m)
        self.freq_ghz = freq_ghz
        self.lx = lx        
        self.lz = lz
        self.dpml = dpml
        self.wind_speed = wind_speed
        self.fetch_km = fetch_km
        self.tx_height = tx_height
        self.rx_height = rx_height
        self.terrain_type = terrain_type
        self.base_water_level = 0# 绝对坐标系下的平均海平面位置
        
        # FDTD 网格分辨率计算
        self.resolution = 10        # FDTD 网格分辨率
        self.dx_fdtd = 1.0 / self.resolution
        
        self.x_full = np.arange(0, self.lx + 2 * self.dpml, self.dx_fdtd)
        self._generate_terrain()

    def _generate_terrain(self):
        """符合 OCP 原则的地形生成工厂"""
        if self.terrain_type == 'flat':
            # 退化验证：平坦海面
            self.h_full = np.zeros_like(self.x_full)
        elif self.terrain_type == 'gaussian_hill':
            # 规范地形验证：高斯山丘 
            hill_center = self.lx / 2.0
            hill_height = 1.0
            hill_width = 20.0
            self.h_full = hill_height * np.exp(-0.5 * ((self.x_full - hill_center) / hill_width)**2)
        else:
            raise ValueError("Unsupported terrain type")


In [23]:
# ==========================================
# 双射线求解接口 (TwoRaySolver)
# ==========================================
class TwoRaySolver:
    """双射线解析模型求解器 (针对 2D 柱面波传播)"""
    def __init__(self, scene: SceneGenerator):
        self.scene = scene
        self.k0 = 2 * np.pi * (scene.freq_ghz * 1e9) / 299792458.0

    def run(self, x_coords, z_coords):
        # 构建计算网格
        X, Z = np.meshgrid(x_coords, z_coords, indexing='ij')
        X_safe = np.maximum(X, 1e-3) # 防止除零
        
        # 直射径与反射径
        R1 = np.sqrt(X_safe**2 + (Z - self.scene.tx_height)**2)
        R2 = np.sqrt(X_safe**2 + (Z + self.scene.tx_height)**2)
        
        # 2D 柱面波解析解 (大参数 Hankel 函数渐近展开)，PEC 下边界反射系数为 -1
        E_2d = np.exp(1j * self.k0 * R1) / np.sqrt(R1) - np.exp(1j * self.k0 * R2) / np.sqrt(R2)
        E_2d_mag = np.abs(E_2d)
        
        return E_2d_mag


In [24]:
# ==========================================
# FDTD 求解器接口 (FDTD Solver)
# ==========================================
class FDTDSolver:
    def __init__(self, scene: SceneGenerator):
        self.scene = scene

    def run(self):
        mp.verbosity(0)
        
        # ── 坐标系说明 ──────────────────────────────────────────────
        # 绝对坐标 (abs): x_full 的原始坐标，范围 [0, lx+2*dpml]
        # Meep 坐标 (meep): 以仿真域中心为原点，= abs - x_center
        # 物理坐标 (phys): 相对于左侧 PML 边界，= abs - dpml
        # ────────────────────────────────────────────────────────────
        x_center = np.mean(self.scene.x_full)   # ≈ (lx + 2*dpml) / 2
        x_meep   = self.scene.x_full - x_center

        # ── 地形几何体 ──────────────────────────────────────────────
        sea_geometry = []
        floor_z = -self.scene.lz / 2 - self.scene.dpml
        for i in range(len(x_meep) - 1):
            z_val1 = min(self.scene.base_water_level + self.scene.h_full[i],
                         self.scene.lz / 2 - self.scene.dpml - 0.5)
            z_val2 = min(self.scene.base_water_level + self.scene.h_full[i + 1],
                         self.scene.lz / 2 - self.scene.dpml - 0.5)
            v1 = mp.Vector3(x_meep[i],     floor_z)
            v2 = mp.Vector3(x_meep[i + 1], floor_z)
            v3 = mp.Vector3(x_meep[i + 1], z_val2)
            v4 = mp.Vector3(x_meep[i],     z_val1)
            sea_geometry.append(mp.Prism([v1, v2, v3, v4],
                                         height=mp.inf, material=mp.metal))

        cell_size      = mp.Vector3(self.scene.lx + 2 * self.scene.dpml,
                                    self.scene.lz + 2 * self.scene.dpml)
        boundary_layers = [mp.PML(self.scene.dpml)]

        # ── 频率转换 ────────────────────────────────────────────────
        c_light      = 299792458.0
        wavelength_m = c_light / (self.scene.freq_ghz * 1e9)
        freq_meep    = 1.0 / wavelength_m

        # ── 源位置（坐标对齐修复保留）───────────────────────────────
        tx_physical_x = 10.0
        tx_abs_x      = self.scene.dpml + tx_physical_x   # 绝对坐标 = 15.0
        tx_meep_x     = tx_abs_x - x_center               # Meep 坐标
        tx_z_meep     = self.scene.base_water_level + self.scene.tx_height

        # ✅ 恢复：各向同性点源（正确的柱面波物理模型）
        sources = [mp.Source(
            mp.ContinuousSource(frequency=freq_meep),
            component=mp.Ez,
            center=mp.Vector3(tx_meep_x, tx_z_meep),
            size=mp.Vector3(0, 0)    # ✅ 点源，不是线源
        )]

        sim = mp.Simulation(
            cell_size=cell_size,
            boundary_layers=boundary_layers,
            geometry=sea_geometry,
            sources=sources,
            resolution=self.scene.resolution,
            force_complex_fields=True
        )

        steady_state_time = self.scene.lx * 5
        print(f"⏳ 开始 FDTD 仿真 (预计达到稳态时间: {steady_state_time})...")
        sim.run(until=steady_state_time)

        # ── 提取场数据 ──────────────────────────────────────────────
        ez_data = sim.get_array(center=mp.Vector3(), size=cell_size, component=mp.Ez)

        # 坐标轴：从 0 到 cell_size.x（绝对坐标）
        x_coords_full = np.linspace(0, cell_size.x, ez_data.shape[0])
        z_coords_full = np.linspace(-cell_size.y / 2, cell_size.y / 2, ez_data.shape[1])

        # ✅ 修复：提取起点 = 源的绝对坐标 tx_abs_x（而非 dpml+tx_physical_x 的旧错误）
        abs_end_x = self.scene.dpml + self.scene.lx
        tx_x_idx  = np.argmin(np.abs(x_coords_full - tx_abs_x))   # ✅ 与源位置对齐
        end_idx   = np.argmin(np.abs(x_coords_full - abs_end_x))
        z_idx     = np.argmin(np.abs(z_coords_full - tx_z_meep))   # 接收高度

        fdtd_range    = x_coords_full[tx_x_idx:end_idx] - x_coords_full[tx_x_idx]
        fdtd_1d_mag   = np.abs(ez_data[tx_x_idx:end_idx, z_idx])
        fdtd_2d_mag   = np.abs(ez_data[tx_x_idx:end_idx, :])
        z_physical_coords = z_coords_full - self.scene.base_water_level

        print(f"✅ FDTD 全波解计算完毕，有效长度: {fdtd_range[-1]:.2f}m")
        
        # ✅ 修复：返回 tx_abs_x，使 PE 侧能精确对齐起点
        return fdtd_range, fdtd_1d_mag, fdtd_2d_mag, z_physical_coords, tx_abs_x

In [25]:
# ==========================================
# PE 求解器接口 (PE Solver)
# ==========================================
class PESolver:
    def __init__(self, scene: SceneGenerator, dx=0.1, dz=0.1):
        self.c = 299792458.0
        self.freq = scene.freq_ghz * 1e9
        self.k0 = 2 * np.pi * self.freq / self.c
        self.dx = dx
        self.dz = dz
        self.max_z = scene.lz
        self.nz = int(self.max_z / dz)
        self.fft_size = 2 * self.nz 
        self.z = np.arange(self.nz) * self.dz
        self.kz = fft.fftfreq(self.fft_size, d=self.dz) * 2 * np.pi
        self.u = np.zeros(self.fft_size, dtype=np.complex128)
        self._setup_absorber()

    def _setup_absorber(self):
        self.absorber = np.ones(self.nz)
        absorb_layer_thickness = int(self.nz * 0.25)
        start_idx = self.nz - absorb_layer_thickness
        window = 0.5 * (1 + np.cos(np.pi * np.arange(absorb_layer_thickness) / absorb_layer_thickness))
        self.absorber[start_idx:] = window


        # 1. 改进 PE 初值场：引入高斯启动器减少近场震荡
    def init_gaussian_source(self, antenna_z_phys, h_surf_0, beam_width=0.5):
        """
        使用高斯 starter 代替硬点源，beam_width 控制波束宽度
        """
        zeta_a = antenna_z_phys - h_surf_0
        # 构造高斯分布
        self.u[:self.nz] = np.exp(-((self.z - zeta_a)**2) / (2 * beam_width**2))
        
        # 进行频谱过滤，滤除不可传播的大角度分量
        kz_filter = np.exp(-(self.kz / (0.9 * self.k0))**10)
        self.u = fft.ifft(fft.fft(self.u) * kz_filter)
    def march(self, x_surf, h_surf, max_range, receiver_z_phys, smooth_window=10):
        print("⏳ 开始 PE 传播步进...")
        h_surf_smoothed = uniform_filter1d(h_surf, size=smooth_window, mode='nearest')
        
        # 初始化：记录 x=0 点
        results_x    = [0.0]
        results_2d   = [np.abs(self.u[:self.nz])]
        h_surf_pe    = [h_surf_smoothed[0]]
        
        idx_rx_0 = int((receiver_z_phys - h_surf_smoothed[0]) / self.dz)
        E0 = np.abs(self.u[idx_rx_0]) if 0 <= idx_rx_0 < self.nz else 1e-12
        results_E_mag = [E0]

        steps = int(max_range / self.dx)
        for s in range(1, steps + 1):
            # ✅ 修复：正确的步进坐标
            x_curr = (s - 1) * self.dx   # 当前步起点
            x_next = s * self.dx          # 当前步终点（记录点）

            z_curr = np.interp(x_curr, x_surf, h_surf_smoothed)
            z_next = np.interp(x_next, x_surf, h_surf_smoothed)
            slope  = (z_next - z_curr) / self.dx
            beta   = np.arctan(slope)

            # --- 边界条件施加 ---
            gamma = -1.0 + 0j
            val_ref    = np.cos(beta)**2 + 0j
            refraction = np.exp(1j * self.k0 * self.dx * (np.sqrt(val_ref) - 1.0))

            self.u[:self.nz] = self.u[:self.nz] * refraction * self.absorber
            self.u[self.nz + 1:] = gamma * self.u[self.nz - 1: 0: -1]
            self.u[0]     *= (1.0 + gamma)
            self.u[self.nz] = 0.0

            # --- 自由空间衍射传播 ---
            k_eff_sq  = (self.k0 * np.cos(beta))**2
            val_diff  = k_eff_sq - self.kz**2 + 0j
            diffraction = np.exp(1j * self.dx * (np.sqrt(val_diff) - self.k0 * np.cos(beta)))

            u_k      = fft.fft(self.u)
            u_k      = u_k * diffraction
            self.u   = fft.ifft(u_k)

            # ✅ 修复：每步只追加一次，统一用 z_next 和 x_next
            E_mag_2d = np.abs(self.u[:self.nz])
            results_x.append(x_next)
            results_2d.append(E_mag_2d)
            h_surf_pe.append(z_next)

            zeta_rx = receiver_z_phys - z_next
            idx = int(zeta_rx / self.dz) if 0 <= zeta_rx < self.max_z else -1
            results_E_mag.append(E_mag_2d[idx] if idx != -1 else 1e-12)

        # ✅ 新增：对1D结果补偿柱面波几何扩展 1/√r
        x_arr    = np.array(results_x)
        E_arr    = np.array(results_E_mag)

        print("✅ PE 传播计算完毕")
        return (x_arr, E_arr,
                np.array(results_2d).T, self.z, np.array(h_surf_pe))

In [26]:

# ==========================================
# 评估器与数学计算工具 (Metrics Evaluator)
# ==========================================
class MetricsEvaluator:

    # 改进对齐函数：让对齐区间更科学
    @staticmethod
    def align_and_convert_to_dB(pe_mag, ref_mag, pe_range, ref_range, calib_start=100.0):
        pe_dB = 20 * np.log10(pe_mag + 1e-12)
        ref_dB = 20 * np.log10(ref_mag + 1e-12)

        # 在 100m 后的稳定区进行均值对齐
        valid_idx = np.where(ref_range > calib_start)[0]
        pe_interp = np.interp(ref_range[valid_idx], pe_range, pe_dB)
        
        offset_dB = np.mean(pe_interp) - np.mean(ref_dB[valid_idx])
        
        return pe_dB, ref_dB + offset_dB, offset_dB

    @staticmethod
    def calc_rmse(pe_dB, fdtd_dB_aligned, pe_range, fdtd_range, min_range=20.0):
        pe_dB_interp = np.interp(fdtd_range, pe_range, pe_dB)
        valid_mask = fdtd_range > min_range
        rmse = np.sqrt(np.mean((pe_dB_interp[valid_mask] - fdtd_dB_aligned[valid_mask])**2))
        return rmse
    @staticmethod
    def calc_rmse_with_protection(pe_dB, fdtd_dB_aligned, pe_range, fdtd_range, 
                                 min_range=50.0, threshold=-65.0):
        """
        零点保护 RMSE 计算：
        1. 避开近场 (min_range)
        2. 忽略低于 threshold 的深零点，防止对数域误差爆炸
        """
        # 插值对齐
        pe_dB_interp = np.interp(fdtd_range, pe_range, pe_dB)
        
        # 构造掩模：同时满足距离要求和能量要求
        # 只有当 FDTD 场强高于阈值时，才计入误差考核
        mask = (fdtd_range > min_range) & (fdtd_dB_aligned > threshold)
        
        if not np.any(mask):
            return 0.0
            
        error_sq = (pe_dB_interp[mask] - fdtd_dB_aligned[mask])**2
        rmse = np.sqrt(np.mean(error_sq))
        return rmse
        
    @staticmethod
    def calc_cumulative_rmse(pe_dB, fdtd_dB_aligned, pe_range, fdtd_range, min_range=20.0):
        pe_dB_interp = np.interp(fdtd_range, pe_range, pe_dB)
        cum_rmse = np.full_like(fdtd_range, np.nan)
        for i in range(len(fdtd_range)):
            if fdtd_range[i] > min_range:
                valid_idx = np.where((fdtd_range > min_range) & (fdtd_range <= fdtd_range[i]))[0]
                if len(valid_idx) > 0:
                    cum_rmse[i] = np.sqrt(np.mean((pe_dB_interp[valid_idx] - fdtd_dB_aligned[valid_idx])**2))
        return cum_rmse
    @staticmethod
    def align_fields_2d(ref_2d_mag, test_2d_mag):
        """对全波解和PE/TwoRay进行全局dB偏置对齐"""
        ref_dB = 20 * np.log10(ref_2d_mag + 1e-12)
        test_dB = 20 * np.log10(test_2d_mag + 1e-12)
        
        # 使用中心区域计算偏置量
        valid_mask = (ref_dB > -60) & (test_dB > -60)
        if np.any(valid_mask):
            offset = np.mean(test_dB[valid_mask]) - np.mean(ref_dB[valid_mask])
        else:
            offset = 0.0
            
        return ref_dB + offset, test_dB

    @staticmethod
    def calc_mbe(pe_dB, fdtd_dB_aligned, pe_range, fdtd_range, min_range=50.0):
        pe_dB_interp = np.interp(fdtd_range, pe_range, pe_dB)
        mask = fdtd_range > min_range
        return np.mean(pe_dB_interp[mask] - fdtd_dB_aligned[mask])


In [27]:
# ==========================================
# 可视化组件层 (Visualizers conforming to OCP)
# ==========================================
class BaseVisualizer(ABC):
    """可视化基类，强制所有子类实现 plot 方法"""
    @abstractmethod
    def plot(self, *args, **kwargs):
        pass

class HeatmapVisualizer(BaseVisualizer):
    def plot(self, ref_2d_dB, test_2d_dB, ref_range, test_range, z_coords, 
             ref_title="Reference 2D Field", test_title="PE 2D Field", save_name="Fig_Heatmap.png"):
        """绘制 2D 空间场强热力图对比"""
        fig = plt.figure(figsize=(12, 8))
        grid = ImageGrid(fig, 111, nrows_ncols=(2, 1), axes_pad=0.4, 
                         share_all=True, cbar_location="right", cbar_mode="single", cbar_pad=0.1)
        
        vmin, vmax = -80, -20 # 统一色标动态范围
        extent_abs = [ref_range[0], ref_range[-1], z_coords[0], z_coords[-1]]
        
        # 绘制参考场 (FDTD 或 Two-Ray)
        im1 = grid[0].imshow(ref_2d_dB.T, extent=extent_abs, origin='lower', aspect='auto', cmap='jet', vmin=vmin, vmax=vmax)
        grid[0].set_title(ref_title)
        grid[0].set_ylabel('Height (m)')
        
        # 绘制测试场 (PE 映射后)
        im2 = grid[1].imshow(test_2d_dB, extent=extent_abs, origin='lower', aspect='auto', cmap='jet', vmin=vmin, vmax=vmax)
        grid[1].set_title(test_title)
        grid[1].set_xlabel('Range (m)')
        grid[1].set_ylabel('Height (m)')
        
        grid[0].cax.colorbar(im1)
        plt.savefig(save_name, dpi=300, bbox_inches='tight')
        plt.close()

class LinePlotVisualizer(BaseVisualizer):
    def plot(self, range_arr, ref_1d_dB, test_1d_dB, ref_label, test_label, rmse_val, title, save_name):
        """绘制 1D 场强切片对比曲线 (Normalized Field Strength)"""
        plt.figure(figsize=(12, 6))
        plt.plot(range_arr, ref_1d_dB, c='g', linestyle='-', alpha=0.6, linewidth=2, label=f'{ref_label}')
        plt.plot(range_arr, test_1d_dB, c='b', linestyle='--', linewidth=2, label=f'{test_label} (RMSE={rmse_val:.2f}dB)')
        # 改进绘图逻辑：在图中画出“无效区”警戒线
        # 在 LinePlotVisualizer.plot 中添加：
        # plt.axvline(x=30, color='r', linestyle=':', label='Near-field Boundary')
        plt.title(title)
        plt.xlabel('Range (m)')
        plt.ylabel('Normalized Field Strength (dB)')
        plt.ylim([-90, -20])
        plt.xlim([range_arr[0], range_arr[-1]])
        plt.grid(True, linestyle=':', alpha=0.7)
        plt.legend(loc='lower left')
        plt.tight_layout()
        plt.savefig(save_name, dpi=300)
        plt.close()

class CumulativeRMSEVisualizer(BaseVisualizer):
    def plot(self, range_arr, cum_rmse_arr, label, save_name):
        """绘制累积 RMSE 收敛曲线"""
        plt.figure(figsize=(10, 5))
        
        valid_mask = ~np.isnan(cum_rmse_arr)
        plt.plot(range_arr[valid_mask], cum_rmse_arr[valid_mask], c='r', linewidth=2, label=label)
        
        plt.title('Cumulative RMSE vs. Range')
        plt.xlabel('Range (m)')
        plt.ylabel('Cumulative RMSE (dB)')
        plt.xlim([range_arr[0], range_arr[-1]])
        plt.grid(True, linestyle=':', alpha=0.7)
        plt.legend()
        plt.tight_layout()
        plt.savefig(save_name, dpi=300)
        plt.close()

In [28]:
# ==========================================
# 主流程控制 (Experiment Execution)
# ==========================================
if __name__ == "__main__":
    
    # 实例化绘图工具
    heatmap_vis = HeatmapVisualizer()
    lineplot_vis = LinePlotVisualizer()
    rmse_vis = CumulativeRMSEVisualizer()

    # ---------------------------------------------------------
    # 实验一：退化验证（平坦海面） PE vs Two-Ray
    # ---------------------------------------------------------
    print("\n" + "="*50)
    print("▶ 开始实验一：退化验证 (PE vs Two-Ray 解析解)")
    print("="*50)
    
    scene_flat = SceneGenerator(terrain_type='flat', lx=300.0)
    
    # PE 求解
    tx_physical_x = 10.0
    tx_idx = np.argmin(np.abs(scene_flat.x_full - tx_physical_x))
    pe_x_input = scene_flat.x_full[tx_idx:] - scene_flat.x_full[tx_idx]
    pe_h_input = scene_flat.h_full[tx_idx:]
    
    pe_solver_flat = PESolver(scene_flat)
    pe_solver_flat.init_gaussian_source(scene_flat.tx_height, pe_h_input[0])
    pe_range_flat, pe_1d_flat, pe_2d_raw_flat, z_coords_pe_flat, h_surf_flat = pe_solver_flat.march(
        pe_x_input, pe_h_input, scene_flat.lx - tx_physical_x, scene_flat.rx_height
    )
    
    # Two-Ray 求解
    tr_solver = TwoRaySolver(scene_flat)
    # 定义绝对物理高度网格 (相对于海平面 0)
    z_coords_abs = np.linspace(scene_flat.base_water_level, scene_flat.base_water_level + scene_flat.lz, pe_2d_raw_flat.shape[0])
    tr_2d_mag = tr_solver.run(pe_range_flat, z_coords_abs)
    
    # 提取 Two-Ray 1D 接收切片
    rx_z_idx = np.argmin(np.abs(z_coords_abs - scene_flat.rx_height))
    tr_1d_flat = tr_2d_mag[:, rx_z_idx]
    
    # 将 PE 2D 场逆映射回绝对物理坐标
    pe_2d_mapped_flat = np.ones((len(z_coords_abs), len(pe_range_flat))) * 1e-12
    for i in range(len(pe_range_flat)):
        z_abs_pe = z_coords_pe_flat + h_surf_flat[i] 
        pe_2d_mapped_flat[:, i] = np.interp(z_coords_abs, z_abs_pe, pe_2d_raw_flat[:, i], left=1e-12, right=1e-12)
        
    # 对齐并计算误差 (Two-Ray vs PE)
    tr_dB_flat, pe_dB_flat = MetricsEvaluator.align_fields_2d(tr_2d_mag, pe_2d_mapped_flat.T)
    pe_1d_dB_flat, tr_1d_dB_aligned, _ = MetricsEvaluator.align_and_convert_to_dB(pe_1d_flat, tr_1d_flat, pe_range_flat, pe_range_flat)
    rmse_flat = MetricsEvaluator.calc_rmse_with_protection(pe_1d_dB_flat, tr_1d_dB_aligned, pe_range_flat, pe_range_flat, min_range=20.0)
    print(f"✅ 平坦海面 RMSE: {rmse_flat:.4f} dB")
    
    # 绘图 - 实验一
    heatmap_vis.plot(tr_dB_flat, pe_dB_flat, pe_range_flat, pe_range_flat, z_coords_abs, 
                     ref_title="Two-Ray Analytical 2D Field", test_title="PE 2D Field (Flat)", save_name="Exp1_Heatmap_Flat.png")
    lineplot_vis.plot(pe_range_flat, tr_1d_dB_aligned, pe_1d_dB_flat, "Two-Ray Analytical", "PE Model", rmse_flat, 
                      "1D Field Comparison (Flat Surface)", "Exp1_LinePlot_Flat.png")


    # ---------------------------------------------------------
    # 实验二：遮挡效应与 PLST 验证（高斯山丘） FDTD vs PE
    # ---------------------------------------------------------
    print("\n" + "="*50)
    print("▶ 开始实验二：PLST 遮挡验证 (PE vs FDTD 全波解)")
    print("="*50)
    
    scene_hill = SceneGenerator(terrain_type='gaussian_hill')
    
    # FDTD 求解
    fdtd_solver = FDTDSolver(scene_hill)
    # FDTD 返回值
    fdtd_range, fdtd_1d_hill, fdtd_2d_hill, z_coords_fdtd, tx_abs_x_hill = fdtd_solver.run()

    # ✅ PE 用 tx_abs_x 定位起点，与 FDTD 源位置严格对齐
    tx_idx_hill = np.argmin(np.abs(scene_hill.x_full - tx_abs_x_hill))
    pe_x_hill   = scene_hill.x_full[tx_idx_hill:] - scene_hill.x_full[tx_idx_hill]
    pe_h_hill   = scene_hill.h_full[tx_idx_hill:]

    # PE 源设置，beam_width 与 FDTD 一致
    wavelength  = 299792458.0 / (scene_hill.freq_ghz * 1e9)
    pe_solver_hill = PESolver(scene_hill)
    pe_solver_hill.init_gaussian_source(
        scene_hill.tx_height, pe_h_hill[0],
        beam_width=0.5 * wavelength    # ✅ 与 FDTD gaussian_amp 一致
    )
    pe_range_hill, pe_1d_hill, pe_2d_raw_hill, z_coords_pe_hill, h_surf_pe_hill = pe_solver_hill.march(
        pe_x_hill, pe_h_hill, fdtd_range[-1], scene_hill.rx_height
    )
    
    # 将 PE 2D 场逆映射回 FDTD 所在的绝对物理坐标
    pe_2d_mapped_hill = np.ones((len(z_coords_fdtd), len(pe_range_hill))) * 1e-12
    for i in range(len(pe_range_hill)):
        z_abs_pe = z_coords_pe_hill + h_surf_pe_hill[i] 
        pe_2d_mapped_hill[:, i] = np.interp(z_coords_fdtd, z_abs_pe, pe_2d_raw_hill[:, i], left=1e-12, right=1e-12)

    # 1D 对齐与 RMSE 计算 (去除前 30m 近场误差)
    x_safe_hill = np.maximum(pe_range_hill, 1e-3)
    pe_1d_hill_compensated = pe_1d_hill / np.sqrt(x_safe_hill)
    pe_1d_dB_hill, fdtd_1d_dB_aligned, offset_hill = \
    MetricsEvaluator.align_and_convert_to_dB(
        pe_1d_hill_compensated, fdtd_1d_hill,
        pe_range_hill, fdtd_range,
        calib_start=220.0
    )
    rmse_hill = MetricsEvaluator.calc_rmse_with_protection(
    pe_1d_dB_hill, fdtd_1d_dB_aligned,
    pe_range_hill, fdtd_range,
    min_range=50.0, threshold=-80.0
    )
    cum_rmse_hill = MetricsEvaluator.calc_cumulative_rmse(pe_1d_dB_hill, fdtd_1d_dB_aligned, pe_range_hill, fdtd_range, min_range=0.0)
    print(f"✅ 高斯山丘 RMSE (忽略近场): {rmse_hill:.4f} dB")
    
    # 2D 空间对齐
    fdtd_2d_dB_aligned = 20 * np.log10(fdtd_2d_hill + 1e-12) + offset_hill
    pe_2d_dB_hill = 20 * np.log10(pe_2d_mapped_hill + 1e-12)
    
    # 绘图 - 实验二
    heatmap_vis.plot(fdtd_2d_dB_aligned, pe_2d_dB_hill, fdtd_range, pe_range_hill, z_coords_fdtd, 
                     ref_title="FDTD Full-Wave 2D Field (Aligned)", test_title="PE (PLST) 2D Field", save_name="Exp2_Heatmap_Hill.png")
    lineplot_vis.plot(fdtd_range, fdtd_1d_dB_aligned, np.interp(fdtd_range, pe_range_hill, pe_1d_dB_hill), 
                      "FDTD Full-Wave", "PE (PLST)", rmse_hill, "1D Field Comparison (Gaussian Hill)", "Exp2_LinePlot_Hill.png")
    rmse_vis.plot(fdtd_range, cum_rmse_hill, "Cum. RMSE vs FDTD", "Exp2_CumRMSE_Hill.png")
    
    print("\n🎉 所有降级验证实验运行完成，图表已保存到当前目录。")


▶ 开始实验一：退化验证 (PE vs Two-Ray 解析解)
⏳ 开始 PE 传播步进...
✅ PE 传播计算完毕
✅ 平坦海面 RMSE: 1.7219 dB

▶ 开始实验二：PLST 遮挡验证 (PE vs FDTD 全波解)
⏳ 开始 FDTD 仿真 (预计达到稳态时间: 1600.0)...


FloatProgress(value=0.0, description='0% done ', max=1600.0)

✅ FDTD 全波解计算完毕，有效长度: 309.89m
⏳ 开始 PE 传播步进...
✅ PE 传播计算完毕
✅ 高斯山丘 RMSE (忽略近场): 1.8802 dB

🎉 所有降级验证实验运行完成，图表已保存到当前目录。
